In [3]:
"""
TASA Silver NAICS/ISIC Emissions Dashboard
Ingests TASA_Silver_NAICS_ISIC.csv and renders an interactive Dash dashboard
with stacked bar charts (Scope 1/2/3) and a 2020-2024 trend view.

Requirements:
    pip install pandas dash plotly
"""
#!pip install panel plotly
#!pip install ipywidgets plotly
#!pip install jupyter_bokeh

import pandas as pd
import numpy as np
import plotly.graph_objects as go



1. Import silver file

In [4]:
def load_data(path: str = "TASA-EFX_KOR_Platinum_Model_2024_v3.0-test.xlsx") -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name="Hotspots_USD", dtype=str, header = 1)
    return df
df = load_data()
df.head()

,Sectors,2017 Total Emissions Intensity,2017 Scope 1,2017 Scope 2,2017 Scope 3,2017 Tier 1 Input Sector,2017 Tier 1 Total Emissions Intensity,2017 Tier 1 Scope 1,2017 Tier 1 Scope 2,2017 Tier 1 Scope 3,...,2023 Tier 1 Scope 3,2024 Total Emissions Intensity,2024 Scope 1,2024 Scope 2,2024 Scope 3,2024 Tier 1 Input Sector,2024 Tier 1 Total Emissions Intensity,2024 Tier 1 Scope 1,2024 Tier 1 Scope 2,2024 Tier 1 Scope 3
0,Rice,2074.823865605154,1749.289237032731,53.73848327539864,271.7961452970246,Fertilizer and Nitrogen Compounds,103.3356462863883,23.39828806783237,15.43919804802246,64.49816017053342,...,48.14246355864405,1723.243731907722,1472.770767244222,36.76698892789958,213.7059757355997,Fertilizer and Nitrogen Compounds,80.36840023316039,19.69960938439024,10.56324609643683,50.10554475233332
1,Rice,2074.823865605154,1749.289237032731,53.73848327539864,271.7961452970246,Rice,40.19788789473291,33.89094072580244,1.04113585839606,5.265811310534404,...,3.971917893497024,1723.243731907722,1472.770767244222,36.76698892789958,213.7059757355997,Rice,33.3863319671831,28.53363853083297,0.7123280793377252,4.140365357012406
2,Rice,2074.823865605154,1749.289237032731,53.73848327539864,271.7961452970246,Pesticides,37.1217321997434,3.168999765664076,1.15556066295351,32.79717177112582,...,24.51797808451581,1723.243731907722,1472.770767244222,36.76698892789958,213.7059757355997,Pesticides,28.98230520229041,2.668060900089168,0.7906156540108051,25.52362864819044
3,Rice,2074.823865605154,1749.289237032731,53.73848327539864,271.7961452970246,"Agriculture, Forestry, and Fishery Service",32.2494770872957,13.37977802391782,5.121632811730922,13.74806625164696,...,10.32240469839763,1723.243731907722,1472.770767244222,36.76698892789958,213.7059757355997,"Agriculture, Forestry, and Fishery Service",25.521718067458,11.2647728738493,3.504137173292432,10.75280802031627
4,Rice,2074.823865605154,1749.289237032731,53.73848327539864,271.7961452970246,"Wholesale, Retail, and Commodity Brokerage Ser...",10.45307798357581,1.492281614572928,2.276651243788025,6.684145125214861,...,4.94123883551092,1723.243731907722,1472.770767244222,36.76698892789958,213.7059757355997,"Wholesale, Retail, and Commodity Brokerage Ser...",7.949260761755868,1.256389562064113,1.557647443156688,5.135223756535067


In [5]:
def filter_wine_2024(df: pd.DataFrame) -> pd.DataFrame:
    # Identify 2024 columns (any column containing "2024")
    cols_2024 = [col for col in df.columns if "2024" in str(col)]
 
    # Keep Sectors column + all 2024 columns
    keep_cols = ["Sectors"] + cols_2024
    df = df[keep_cols]
 
    # Filter rows to wine sector only
    df = df[df["Sectors"] == "Abrasive Material"].copy()
 
    # Rename sector for display
    df["Sectors"] = "Abrasive Material"
 
    # Convert numeric columns from str to float, skipping known string columns
    string_cols = ["2024 Tier 1 Input Sector"]
    for col in cols_2024:
        if col not in string_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
 
    return df.reset_index(drop=True)

df_wine = filter_wine_2024(df)
df_wine.head()

,Sectors,2024 Total Emissions Intensity,2024 Scope 1,2024 Scope 2,2024 Scope 3,2024 Tier 1 Input Sector,2024 Tier 1 Total Emissions Intensity,2024 Tier 1 Scope 1,2024 Tier 1 Scope 2,2024 Tier 1 Scope 3
0,Abrasive Material,477.516714,43.48562,57.259274,376.77182,Basic Inorganic Compounds,39.807354,5.911731,12.008619,21.887003
1,Abrasive Material,477.516714,43.48562,57.259274,376.77182,Other Fiber Fabrics,37.578098,3.581560,5.924311,28.072227
2,Abrasive Material,477.516714,43.48562,57.259274,376.77182,Adhesive and Gelatin,25.649723,1.771276,1.490530,22.387917
3,Abrasive Material,477.516714,43.48562,57.259274,376.77182,Synthetic Resin,25.157853,4.960154,3.017311,17.180389
4,Abrasive Material,477.516714,43.48562,57.259274,376.77182,Manufacturing Equipment Repair,24.354784,2.189486,3.682700,18.482598


In [9]:
"""
Wine Baseline Emissions Structure Chart — polished, interactive
Toggle: Tier 1 Scope broken out (S1/S2/S3 per sector) vs summed (3 global buckets)
Assumes df_wine is already loaded and filtered.
"""

import json
import pandas as pd
from IPython.display import display, HTML

# ─────────────────────────────────────────────────────────────────────────────
# 1. EXTRACT DATA
# ─────────────────────────────────────────────────────────────────────────────
first_row = df_wine.iloc[0]
s1_total  = float(first_row["2024 Scope 1"])
s2_total  = float(first_row["2024 Scope 2"])
s3_total  = float(first_row["2024 Scope 3"])
total_ei  = float(first_row["2024 Total Emissions Intensity"])

tier1_raw = (
    df_wine[[
        "2024 Tier 1 Input Sector",
        "2024 Tier 1 Total Emissions Intensity",
        "2024 Tier 1 Scope 1",
        "2024 Tier 1 Scope 2",
        "2024 Tier 1 Scope 3",
    ]]
    .drop_duplicates()
    .sort_values("2024 Tier 1 Total Emissions Intensity", ascending=False)
    .reset_index(drop=True)
)

is_other  = tier1_raw["2024 Tier 1 Input Sector"].str.strip() == "Other"
non_other = tier1_raw[~is_other].sort_values("2024 Tier 1 Total Emissions Intensity", ascending=False)
other     = tier1_raw[is_other].sort_values("2024 Tier 1 Total Emissions Intensity", ascending=False)
tier1     = pd.concat([non_other, other]).reset_index(drop=True)

scope_data = [
    {"label": "Scope 1", "val": round(s1_total,2), "pct": round(s1_total/total_ei*100,1), "color": "#3A86C8"},
    {"label": "Scope 2", "val": round(s2_total,2), "pct": round(s2_total/total_ei*100,1), "color": "#2BAE96"},
    {"label": "Scope 3", "val": round(s3_total,2), "pct": round(s3_total/total_ei*100,1), "color": "#1B4F8A"},
]

tier1_data = []
for _, row in tier1.iterrows():
    t_total = float(row["2024 Tier 1 Total Emissions Intensity"])
    t_s1    = float(row["2024 Tier 1 Scope 1"])
    t_s2    = float(row["2024 Tier 1 Scope 2"])
    t_s3    = float(row["2024 Tier 1 Scope 3"])
    tier1_data.append({
        "label":  str(row["2024 Tier 1 Input Sector"]),
        "val":    round(t_total,2), "pct": round(t_total/total_ei*100,1),
        "s1_val": round(t_s1,2),   "s1":  round(t_s1/total_ei*100,1),
        "s2_val": round(t_s2,2),   "s2":  round(t_s2/total_ei*100,1),
        "s3_val": round(t_s3,2),   "s3":  round(t_s3/total_ei*100,1),
    })

# Summed mode: aggregate S1/S2/S3 across all T1 sectors
sum_s1 = sum(r["s1_val"] for r in tier1_data)
sum_s2 = sum(r["s2_val"] for r in tier1_data)
sum_s3 = sum(r["s3_val"] for r in tier1_data)
summed_data = [
    {"label": "Scope 1", "val": round(sum_s1,2), "pct": round(sum_s1/total_ei*100,1), "color": "#3A86C8"},
    {"label": "Scope 2", "val": round(sum_s2,2), "pct": round(sum_s2/total_ei*100,1), "color": "#2BAE96"},
    {"label": "Scope 3", "val": round(sum_s3,2), "pct": round(sum_s3/total_ei*100,1), "color": "#1B4F8A"},
]

payload = json.dumps({
    "total_ei": round(total_ei,2),
    "scopes":   scope_data,
    "tier1":    tier1_data,
    "summed":   summed_data,
})

html = f"""
<div style="display:inline-block;font-family:'Segoe UI',Arial,sans-serif;">

  <!-- Toggle buttons -->
  <div style="margin-bottom:10px;display:flex;gap:8px;align-items:center;">
    <span style="font-size:11px;color:#4A6A8A;font-weight:600;margin-right:4px;">Tier 1 Scope view:</span>
    <button id="btn-detail-643dc346" onclick="setMode_643dc346('detail')"
      style="padding:5px 14px;border-radius:20px;border:none;cursor:pointer;font-size:11px;
             font-family:'Segoe UI',Arial,sans-serif;font-weight:600;
             background:#1B4F8A;color:#fff;box-shadow:0 1px 4px rgba(0,0,0,0.15);">
      Broken Out
    </button>
    <button id="btn-summed-643dc346" onclick="setMode_643dc346('summed')"
      style="padding:5px 14px;border-radius:20px;border:2px solid #1B4F8A;cursor:pointer;font-size:11px;
             font-family:'Segoe UI',Arial,sans-serif;font-weight:600;
             background:#fff;color:#1B4F8A;">
      Summed
    </button>
  </div>

  <div style="position:relative;display:inline-block;
              background:linear-gradient(135deg,#e8f0f7 0%,#f0f5f9 100%);
              padding:20px 28px 28px;border-radius:12px;
              box-shadow:0 2px 14px rgba(0,0,0,0.09);">
    <canvas id="wesc-canvas-643dc346" style="display:block;"></canvas>
    <div id="wesc-tip-643dc346" style="
      display:none;position:fixed;pointer-events:none;
      background:rgba(10,30,55,0.92);color:#fff;
      padding:8px 12px;border-radius:6px;font-size:12px;
      box-shadow:0 3px 12px rgba(0,0,0,0.3);
      line-height:1.6;z-index:9999;max-width:220px;"></div>
  </div>
</div>

<script>
setTimeout(function(){{
const DATA = {payload};
let currentMode = 'detail';

// ── Layout ────────────────────────────────────────────────────────────────
const TOP     = 72;
const LPAD    = 16;
const FULL_H  = 900;
const BOX_GAP = 3;
const MIN_BOX = 12;

const ANC_W = 120, SC_W = 110, T1_W = 240, LF_W = 110, GAP = 144;
const ANC_X = LPAD;
const SC_X  = ANC_X + ANC_W + GAP;
const T1_X  = SC_X  + SC_W  + GAP;
const LF_X  = T1_X  + T1_W  + GAP;
const TOTAL_W = LF_X + LF_W + LPAD + 10;
const TOTAL_H = TOP + FULL_H + 32;

const C   = document.getElementById('wesc-canvas-643dc346');
const DPR = window.devicePixelRatio || 1;
C.width   = TOTAL_W * DPR;
C.height  = TOTAL_H * DPR;
C.style.width  = TOTAL_W + 'px';
C.style.height = TOTAL_H + 'px';
const ctx = C.getContext('2d');
ctx.scale(DPR, DPR);

const S_COLORS  = ['#3A86C8','#2BAE96','#1B4F8A'];
const T1_COLORS = [
  '#27AE8F','#2E86C1','#1D8A6E','#2471A3','#1A7A60',
  '#1F6FA3','#17725A','#1A5F8C','#136150','#174F78',
  '#0F5143','#143F65','#0D4438','#103356','#0A3830',
  '#0C2B48','#082E27','#09223C','#062420','#071B30','#051510'
];

// ── Helpers ───────────────────────────────────────────────────────────────
function hexToRgb(hex) {{
  return [parseInt(hex.slice(1,3),16),parseInt(hex.slice(3,5),16),parseInt(hex.slice(5,7),16)];
}}
function rgbaStr(hex,a) {{
  const [r,g,b]=hexToRgb(hex); return `rgba(${{r}},${{g}},${{b}},${{a}})`;
}}
function rRect(x,y,w,h,r) {{ ctx.beginPath(); ctx.roundRect(x,y,w,h,r); }}

function proportionalHeights(pcts, totalH, gap) {{
  const n=pcts.length, tot=pcts.reduce((a,b)=>a+b,0)||1;
  const available=totalH-gap*(n-1);
  let heights=pcts.map(p=>Math.max(MIN_BOX,(p/tot)*available));
  let diff=heights.reduce((a,b)=>a+b,0)-available;
  if (Math.abs(diff)>0.5) {{
    const adj=heights.map((h,i)=>{{return{{i,h}}}}).filter(o=>o.h>MIN_BOX).sort((a,b)=>b.h-a.h);
    for (const o of adj) {{
      const cut=Math.min(Math.abs(diff),heights[o.i]-MIN_BOX);
      heights[o.i]-=Math.sign(diff)*cut; diff-=Math.sign(diff)*cut;
      if (Math.abs(diff)<0.5) break;
    }}
  }}
  return heights;
}}

function drawBox(x,y,w,h,color,label,pct,val,fs) {{
  fs=fs||9;
  const [r,g,b]=hexToRgb(color);
  const grad=ctx.createLinearGradient(x,y,x,y+h);
  grad.addColorStop(0,`rgba(${{Math.min(r+30,255)}},${{Math.min(g+30,255)}},${{Math.min(b+30,255)}},1)`);
  grad.addColorStop(1,color);
  ctx.fillStyle=grad; rRect(x,y,w,h,4); ctx.fill();
  const hi=ctx.createLinearGradient(x,y,x,y+Math.min(h*0.5,18));
  hi.addColorStop(0,'rgba(255,255,255,0.18)'); hi.addColorStop(1,'rgba(255,255,255,0)');
  ctx.fillStyle=hi; rRect(x,y,w,h,4); ctx.fill();
  ctx.strokeStyle=`rgba(${{Math.max(r-20,0)}},${{Math.max(g-20,0)}},${{Math.max(b-20,0)}},0.3)`;
  ctx.lineWidth=0.7; rRect(x,y,w,h,4); ctx.stroke();
  if(h<8) return;
  ctx.textBaseline='middle'; ctx.textAlign='left';
  const px=x+6, maxW=w-10;
  if(h>=22) {{
    ctx.font=`600 ${{fs}}px 'Segoe UI',Arial,sans-serif`;
    ctx.fillStyle='rgba(255,255,255,0.95)';
    let txt=label;
    while(ctx.measureText(txt).width>maxW&&txt.length>3) txt=txt.slice(0,-1);
    if(txt!==label) txt=txt.slice(0,-1)+'…';
    ctx.fillText(txt,px,y+h*0.35);
    ctx.font=`${{fs-1}}px 'Segoe UI',Arial,sans-serif`;
    ctx.fillStyle='rgba(255,255,255,0.70)';
    ctx.fillText(pct+'%',px,y+h*0.68);
  }} else {{
    ctx.font=`${{fs-1}}px 'Segoe UI',Arial,sans-serif`;
    ctx.fillStyle='rgba(255,255,255,0.88)';
    ctx.fillText(pct+'%',px,y+h/2);
  }}
}}

function drawRibbon(x1,y1t,y1b,x2,y2t,y2b,color,alpha) {{
  const mx=x1+(x2-x1)*0.52;
  ctx.beginPath();
  ctx.moveTo(x1,y1t); ctx.bezierCurveTo(mx,y1t,mx,y2t,x2,y2t);
  ctx.lineTo(x2,y2b); ctx.bezierCurveTo(mx,y2b,mx,y1b,x1,y1b);
  ctx.closePath(); ctx.fillStyle=rgbaStr(color,alpha); ctx.fill();
}}

function drawHeader(x,w,l1,l2) {{
  ctx.textAlign='center'; ctx.textBaseline='middle';
  ctx.fillStyle='#0D2F4F';
  ctx.font="700 11px 'Segoe UI',Arial,sans-serif";
  ctx.fillText(l1,x+w/2,TOP-(l2?24:14));
  if(l2) {{ ctx.font="400 9px 'Segoe UI',Arial,sans-serif"; ctx.fillStyle='#4A6A8A'; ctx.fillText(l2,x+w/2,TOP-11); }}
  ctx.strokeStyle='rgba(26,63,100,0.15)'; ctx.lineWidth=1;
  ctx.beginPath(); ctx.moveTo(x+4,TOP-3); ctx.lineTo(x+w-4,TOP-3); ctx.stroke();
}}

// ── Tooltip ───────────────────────────────────────────────────────────────
const tip = document.getElementById('wesc-tip-643dc346');
let hitBoxes = [];
function regHit(x,y,w,h,label,pct,val) {{
  hitBoxes.push({{x,y,w,h,tip:`<b>${{label}}</b><br>${{pct}}% of total<br>${{val.toLocaleString()}} Mg CO₂e / M USD`}});
}}
C.addEventListener('mousemove',e=>{{
  const cr=C.getBoundingClientRect();
  const mx=(e.clientX-cr.left)*(TOTAL_W/cr.width);
  const my=(e.clientY-cr.top)*(TOTAL_H/cr.height);
  const hit=hitBoxes.find(b=>mx>=b.x&&mx<=b.x+b.w&&my>=b.y&&my<=b.y+b.h);
  if(hit) {{
    tip.innerHTML=hit.tip; tip.style.display='block';
    tip.style.left=(e.clientX+14)+'px'; tip.style.top=(e.clientY-10)+'px';
  }} else tip.style.display='none';
}});
C.addEventListener('mouseleave',()=>tip.style.display='none');

// ── Button state ──────────────────────────────────────────────────────────
window.setMode_643dc346 = function(mode) {{
  currentMode = mode;
  document.getElementById('btn-detail-643dc346').style.background  = mode==='detail'  ? '#1B4F8A' : '#fff';
  document.getElementById('btn-detail-643dc346').style.color       = mode==='detail'  ? '#fff'    : '#1B4F8A';
  document.getElementById('btn-detail-643dc346').style.border      = mode==='detail'  ? 'none'    : '2px solid #1B4F8A';
  document.getElementById('btn-summed-643dc346').style.background  = mode==='summed' ? '#1B4F8A' : '#fff';
  document.getElementById('btn-summed-643dc346').style.color       = mode==='summed' ? '#fff'    : '#1B4F8A';
  document.getElementById('btn-summed-643dc346').style.border      = mode==='summed' ? 'none'    : '2px solid #1B4F8A';
  render(mode);
}};

// ── Main render ───────────────────────────────────────────────────────────
function render(mode) {{
  hitBoxes = [];
  ctx.clearRect(0,0,TOTAL_W,TOTAL_H);

  // background
  const bg=ctx.createLinearGradient(0,0,0,TOTAL_H);
  bg.addColorStop(0,'rgba(232,240,247,0)'); bg.addColorStop(1,'rgba(240,245,249,0)');
  ctx.fillStyle=bg; ctx.fillRect(0,0,TOTAL_W,TOTAL_H);

  // headers
  drawHeader(ANC_X,ANC_W,'BASELINE','EMISSIONS STRUCTURE');
  drawHeader(SC_X, SC_W, 'SCOPE',   null);
  drawHeader(T1_X, T1_W, 'TIER 1',  'Input Sector');
  drawHeader(LF_X, LF_W, mode==='detail' ? 'TIER 1 SCOPE' : 'TIER 1 SCOPE', mode==='detail'?'by sector':'summed');

  // ── Anchor chevron
  const chev=18;
  const aGrad=ctx.createLinearGradient(ANC_X,TOP,ANC_X,TOP+FULL_H);
  aGrad.addColorStop(0,'#1A4A72'); aGrad.addColorStop(1,'#0D2F4F');
  ctx.fillStyle=aGrad;
  ctx.beginPath();
  ctx.moveTo(ANC_X,TOP); ctx.lineTo(ANC_X+ANC_W-chev,TOP);
  ctx.lineTo(ANC_X+ANC_W,TOP+FULL_H/2); ctx.lineTo(ANC_X+ANC_W-chev,TOP+FULL_H);
  ctx.lineTo(ANC_X,TOP+FULL_H); ctx.closePath(); ctx.fill();
  const ancCX=ANC_X+(ANC_W-chev)/2;
  ctx.textAlign='center'; ctx.textBaseline='middle';
  ctx.font="700 14px 'Segoe UI',Arial,sans-serif"; ctx.fillStyle='#fff';
  ctx.fillText('Abrasive Material',ancCX,TOP+FULL_H*0.40);
  ctx.font="400 9px 'Segoe UI',Arial,sans-serif"; ctx.fillStyle='rgba(255,255,255,0.65)';
  ctx.fillText(DATA.total_ei.toLocaleString()+' Mg CO₂e',ancCX,TOP+FULL_H*0.53);
  regHit(ANC_X,TOP,ANC_W,FULL_H,'Abrasive Material Sector',100,DATA.total_ei);

  // ── Scope column
  const scPcts=DATA.scopes.map(s=>s.pct);
  const scHeights=proportionalHeights(scPcts,FULL_H,BOX_GAP);
  let scY_arr=[]; let sy=TOP;
  scHeights.forEach(h=>{{scY_arr.push(sy);sy+=h+BOX_GAP;}});

  // anchor→scope ribbons (S3 behind, S1/S2 in front)
  const ancSlices=[]; let ancCursor=TOP;
  DATA.scopes.forEach((sc,si)=>{{
    const share=sc.pct/scPcts.reduce((a,b)=>a+b,0);
    ancSlices.push({{top:ancCursor,bot:ancCursor+share*FULL_H,scY:scY_arr[si],h:scHeights[si],color:sc.color}});
    ancCursor+=share*FULL_H;
  }});
  [2,1,0].forEach(si=>{{
    const sl=ancSlices[si];
    drawRibbon(ANC_X+ANC_W-chev,sl.top,sl.bot,SC_X,sl.scY,sl.scY+sl.h,sl.color,0.22);
  }});
  const scopeBoxes=[];
  DATA.scopes.forEach((sc,si)=>{{
    const h=scHeights[si],scY=scY_arr[si];
    scopeBoxes.push({{y:scY,h,mid:scY+h/2,color:sc.color,pct:sc.pct,val:sc.val}});
    drawBox(SC_X,scY,SC_W,h,sc.color,sc.label,sc.pct,sc.val,10);
    regHit(SC_X,scY,SC_W,h,sc.label,sc.pct,sc.val);
  }});

  // ── Tier 1 column
  const t1Pcts=DATA.tier1.map(t=>t.pct);
  const t1Heights=proportionalHeights(t1Pcts,FULL_H,BOX_GAP);
  const s3=scopeBoxes[2];
  let t1Boxes=[]; let t1Y=TOP;

  // Scope3→T1 ribbons (stacked)
  let s3cursor=s3.y;
  DATA.tier1.forEach((t,i)=>{{
    const h=t1Heights[i],col=T1_COLORS[i%T1_COLORS.length];
    const sliceH=(t.pct/(s3.pct||1))*s3.h;
    drawRibbon(SC_X+SC_W,s3cursor,s3cursor+sliceH,T1_X,t1Y,t1Y+h,col,0.20);
    s3cursor+=sliceH; t1Y+=h+BOX_GAP;
  }});
  t1Y=TOP;
  DATA.tier1.forEach((t,i)=>{{
    const h=t1Heights[i],col=T1_COLORS[i%T1_COLORS.length];
    t1Boxes.push({{y:t1Y,h,mid:t1Y+h/2,color:col,pct:t.pct,val:t.val}});
    drawBox(T1_X,t1Y,T1_W,h,col,t.label,t.pct,t.val,9);
    regHit(T1_X,t1Y,T1_W,h,t.label,t.pct,t.val);
    t1Y+=h+BOX_GAP;
  }});

  // ── Leaf column — two modes
  if (mode === 'detail') {{
    // All N*3 leaves share FULL_H proportionally by absolute pct
    const LEAF_GAP=2;
    const allLeafPcts=[];
    DATA.tier1.forEach(t=>allLeafPcts.push(t.s1,t.s2,t.s3));
    const leafTotal=allLeafPcts.reduce((a,b)=>a+b,0)||1;
    const N_LEAVES=allLeafPcts.length;
    const LEAF_AVAIL=FULL_H-LEAF_GAP*(N_LEAVES-1);
    const MIN_LEAF=Math.max(4,LEAF_AVAIL/N_LEAVES*0.25);
    let allLeafH=allLeafPcts.map(p=>Math.max(MIN_LEAF,(p/leafTotal)*LEAF_AVAIL));
    const lsum=allLeafH.reduce((a,b)=>a+b,0);
    allLeafH=allLeafH.map(h=>h/lsum*LEAF_AVAIL);

    const leafGroups=[];
    let flatIdx=0;
    DATA.tier1.forEach((t,i)=>{{
      const tb=t1Boxes[i];
      leafGroups.push({{
        tb, i,
        lPcts:[t.s1,t.s2,t.s3],
        lVals:[t.s1_val,t.s2_val,t.s3_val],
        lh:[allLeafH[flatIdx],allLeafH[flatIdx+1],allLeafH[flatIdx+2]]
      }});
      flatIdx+=3;
    }});
    let leafY=TOP;
    leafGroups.forEach(g=>{{ g.startY=leafY; g.lh.forEach(h=>{{leafY+=h+LEAF_GAP;}}); }});

    // ribbons first
    leafGroups.forEach(g=>{{
      let t1cur=g.tb.y, ly=g.startY;
      const ptot=g.lPcts.reduce((a,b)=>a+b,0)||1;
      g.lh.forEach((lh,si)=>{{
        const sliceH=(g.lPcts[si]/ptot)*g.tb.h;
        drawRibbon(T1_X+T1_W,t1cur,t1cur+sliceH,LF_X,ly,ly+lh,S_COLORS[si],0.20);
        t1cur+=sliceH; ly+=lh+LEAF_GAP;
      }});
    }});
    // boxes
    leafGroups.forEach(g=>{{
      let ly=g.startY;
      g.lh.forEach((lh,si)=>{{
        const lbl=['Scope 1','Scope 2','Scope 3'][si];
        drawBox(LF_X,ly,LF_W,lh,S_COLORS[si],lbl,g.lPcts[si],g.lVals[si],8);
        regHit(LF_X,ly,LF_W,lh,DATA.tier1[g.i].label+' · '+lbl,g.lPcts[si],g.lVals[si]);
        ly+=lh+LEAF_GAP;
      }});
    }});

  }} else {{
    // Summed mode: 3 boxes (S1, S2, S3) spanning full height
    const smPcts=DATA.summed.map(s=>s.pct);
    const smHeights=proportionalHeights(smPcts,FULL_H,BOX_GAP);
    let smY=TOP;

    // T1→summed ribbons: each T1 box fans into 3 summed buckets
    // Draw ribbons first, stacked from T1 right edge
    DATA.tier1.forEach((t,i)=>{{
      const tb=t1Boxes[i];
      const lPcts=[t.s1,t.s2,t.s3];
      const ptot=lPcts.reduce((a,b)=>a+b,0)||1;
      let t1cur=tb.y;
      // target Y for each summed bucket (precompute)
      let bucketY=TOP;
      const bucketTops=smHeights.map(h=>{{const y=bucketY;bucketY+=h+BOX_GAP;return y;}});
      lPcts.forEach((p,si)=>{{
        const sliceH=(p/ptot)*tb.h;
        // destination: proportional slice within the summed bucket
        const bucketH=smHeights[si];
        const bucketTot=DATA.summed[si].pct||1;
        // each T1 sector contributes p% to its bucket slice
        // stack within bucket by T1 order — use a cursor tracked per bucket
        drawRibbon(T1_X+T1_W,t1cur,t1cur+sliceH,
                   LF_X,bucketTops[si],bucketTops[si]+bucketH,
                   S_COLORS[si],0.12);
        t1cur+=sliceH;
      }});
    }});

    // summed boxes on top
    smY=TOP;
    DATA.summed.forEach((sm,si)=>{{
      const h=smHeights[si];
      drawBox(LF_X,smY,LF_W,h,sm.color,sm.label,sm.pct,sm.val,10);
      regHit(LF_X,smY,LF_W,h,'Tier 1 '+sm.label+' (all sectors)',sm.pct,sm.val);
      smY+=h+BOX_GAP;
    }});
  }}

  // footer
  ctx.font="400 8px 'Segoe UI',Arial,sans-serif";
  ctx.fillStyle='rgba(80,110,140,0.60)';
  ctx.textAlign='left'; ctx.textBaseline='top';
  ctx.fillText('Emissions Intensity (Mg CO₂e per Million USD) · 2024',ANC_X,TOP+FULL_H+10);
}}

render('detail');
}}, 100);
</script>
"""

display(HTML(html))